[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coopster-seclusion/sar-optical-pipeline/blob/main/notebooks/04_build_stacks.ipynb)

# Notebook 04: Build Multi-Temporal Stacks

Loads per-epoch GeoTIFFs produced by notebooks 01–03, builds labelled time-axis stacks,
aligns Sentinel-2 to the SAR pixel grids, validates, and saves everything as NetCDF
for use by notebook 05 (change detection).

All epoch lists, polarizations, CRS, and paths come from `config.yaml` or are
auto-detected from available files — no hardcoded values in this notebook.

**Prerequisites:** Run notebooks 01, 02, and 03 first (or at least those sensors you
want to include in the stack).

**Outputs written to `data/stacks/`:**
- `opera_{pol}.nc` — OPERA RTC-S1 stack (time × y × x), one per polarization
- `hyp3_{pol}.nc` — HyP3 RTC-S1 stack (time × y × x), one per polarization
- `s2_indices.nc` — Sentinel-2 NDVI/NDWI/NDBI/BSI Dataset at native resolution
- `s2_at_opera.nc` — S2 indices resampled to the OPERA 30 m grid
- `s2_at_hyp3.nc` — S2 indices resampled to the HyP3 10 m grid

## Setup

In [ ]:
# ── Cell 1: run this first, every session ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = 'https://github.com/coopster-seclusion/sar-optical-pipeline.git'
REPO_DIR = '/content/sar-optical-pipeline'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'rioxarray', 'netcdf4', 'pyogrio'], check=True)

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
import matplotlib.pyplot as plt

from pipeline.env import setup, repo_root
from pipeline.validate import validate_config
from pipeline.utils import resolve_crs
import pipeline.stack as stack_lib

cfg, DATA, OUT = setup(colab=True)
validate_config(cfg, repo_root())

SITE       = cfg['site_name']
OUT_FIGS   = OUT / 'figures'
STACKS_DIR = DATA / 'stacks'
ROOT       = repo_root()
aoi_path   = ROOT / cfg['aoi']['change_area']
crs        = resolve_crs(cfg, aoi_path)

all_years = [e['year'] for e in cfg['epochs']]
baseline  = cfg['change_detection']['baseline_year']
pols      = cfg['opera'].get('polarizations') or ['HH', 'HV']

# ── Optional epoch filter ──────────────────────────────────────────────────────────────
# Set to None to build stacks from all epochs that have available data.
# Set to a list (e.g. [2016, 2022, 2025]) to limit to specific years.
EPOCH_SUBSET = None
candidate_years = [y for y in all_years if EPOCH_SUBSET is None or y in EPOCH_SUBSET]

print(f'Site            : {SITE}')
print(f'Baseline year   : {baseline}')
print(f'Polarizations   : {pols}')
print(f'Output CRS      : {crs}')
print(f'Candidate epochs: {candidate_years}')
print(f'Stacks dir      : {STACKS_DIR}')

## Scan available data files

Each stack is built only from epochs that have all required files on Drive.
Missing files do not raise an error here — the sensor is simply skipped.

In [ ]:
# ── Cell 2: scan for available per-epoch files ────────────────────────────────────
# OPERA: data/opera_rtc/opera_{year}_{pol}.tif — all pols must be present
opera_years = [
    y for y in candidate_years
    if all((DATA / 'opera_rtc' / f'opera_{y}_{p}.tif').exists() for p in pols)
]

# HyP3: data/hyp3/hyp3_{year}_{pol}.tif — all pols must be present
hyp3_years = [
    y for y in candidate_years
    if all((DATA / 'hyp3' / f'hyp3_{y}_{p}.tif').exists() for p in pols)
]

# S2: data/s2/s2_{year}.tif
s2_years = [
    y for y in candidate_years
    if (DATA / 's2' / f's2_{y}.tif').exists()
]

print(f'OPERA epochs with data : {opera_years}')
print(f'HyP3  epochs with data : {hyp3_years}')
print(f'S2    epochs with data : {s2_years}')

for sensor, years in [('OPERA', opera_years), ('HyP3', hyp3_years), ('S2', s2_years)]:
    if years and baseline not in years:
        print(f'WARNING: baseline year {baseline} not in {sensor} data. '
              f'Change detection requires the baseline epoch — '
              f're-run notebook {"02" if sensor == "OPERA" else "03" if sensor == "HyP3" else "01"} '
              f'for year {baseline}.')

if not opera_years and not hyp3_years and not s2_years:
    raise RuntimeError('No data found on Drive. Run notebooks 01, 02, and 03 first.')

## Build SAR stacks

In [ ]:
# ── Cell 3: build OPERA stacks (one DataArray per polarization) ───────────────────
opera_stacks = {}

if not opera_years:
    print('No OPERA data found — skipping. Run notebook 02 first.')
else:
    for pol in pols:
        da = stack_lib.build_sar_stack(DATA, sensor='opera', pol=pol, epochs=opera_years)
        opera_stacks[pol] = da
        print(f'OPERA {pol}: shape={da.shape}  CRS={da.rio.crs}  '
              f'range=[{float(np.nanmin(da)):.1f}, {float(np.nanmax(da)):.1f}] dB')
    print(f'OPERA epochs in stack: {opera_years}')

In [ ]:
# ── Cell 4: build HyP3 stacks (one DataArray per polarization) ────────────────────
hyp3_stacks = {}

if not hyp3_years:
    print('No HyP3 data found — skipping. Run notebook 03 first.')
else:
    for pol in pols:
        da = stack_lib.build_sar_stack(DATA, sensor='hyp3', pol=pol, epochs=hyp3_years)
        hyp3_stacks[pol] = da
        print(f'HyP3 {pol}: shape={da.shape}  CRS={da.rio.crs}  '
              f'range=[{float(np.nanmin(da)):.1f}, {float(np.nanmax(da)):.1f}] dB')
    print(f'HyP3 epochs in stack: {hyp3_years}')

## Build Sentinel-2 spectral-index stack

Reads the 9-band GeoTIFF produced by notebook 01 and extracts the four spectral index
bands (NDVI, NDWI, NDBI, BSI) into a labelled xarray Dataset.

In [ ]:
# ── Cell 5: build S2 spectral-index stack ─────────────────────────────────────────
s2_stack = None

if not s2_years:
    print('No S2 data found — skipping. Run notebook 01 first.')
else:
    s2_stack = stack_lib.build_s2_stack(DATA, epochs=s2_years)
    print(f'S2 indices : {list(s2_stack.data_vars)}')
    print(f'S2 dims    : {dict(s2_stack.dims)}')
    print(f'S2 epochs  : {s2_years}')
    for var in s2_stack.data_vars:
        arr = s2_stack[var].values
        print(f'  {var}: range=[{np.nanmin(arr):.3f}, {np.nanmax(arr):.3f}]')

## Align Sentinel-2 to SAR pixel grids

S2 GEE exports are typically ~10 m. OPERA is 30 m; HyP3 is 10 m. Resampling to the
coarser OPERA grid is preferred for pixel-level change analysis alongside OPERA data.

The aligned stacks share the same (y, x) pixel grid as the SAR data but retain
the S2 time axis — epoch matching is handled in notebook 05.

In [ ]:
# ── Cell 6: resample S2 to SAR pixel grids ────────────────────────────────────────
s2_at_opera = None
s2_at_hyp3  = None

if s2_stack is None:
    print('S2 stack not built — skipping alignment.')
else:
    if opera_stacks:
        # Use a single 2-D slice as the spatial reference — no time dimension
        ref_opera   = next(iter(opera_stacks.values())).isel(time=0)
        s2_at_opera = stack_lib.align_to_grid(s2_stack, ref_opera, resampling='bilinear')
        print(f'S2 → OPERA grid: {dict(s2_at_opera.dims)}  '
              f'pixel={float(abs(s2_at_opera.rio.resolution()[0])):.0f} m')

    if hyp3_stacks:
        ref_hyp3   = next(iter(hyp3_stacks.values())).isel(time=0)
        s2_at_hyp3 = stack_lib.align_to_grid(s2_stack, ref_hyp3, resampling='bilinear')
        print(f'S2 → HyP3  grid: {dict(s2_at_hyp3.dims)}  '
              f'pixel={float(abs(s2_at_hyp3.rio.resolution()[0])):.0f} m')

    if not opera_stacks and not hyp3_stacks:
        print('No SAR stacks built — cannot determine target grid. '
              'Run notebooks 02 or 03 first.')

## Validate stacks

Checks epoch count, CRS presence, and absence of all-NaN time slices.
A validation failure indicates a broken or incomplete input file.

In [ ]:
# ── Cell 7: validate all stacks ────────────────────────────────────────────────────
errors = []

for pol, da in opera_stacks.items():
    try:
        stack_lib.validate_stack(da, opera_years, label=f'OPERA {pol}')
        print(f'OPERA {pol} : OK  ({len(opera_years)} epoch(s))')
    except ValueError as e:
        print(f'OPERA {pol} : FAIL — {e}')
        errors.append(str(e))

for pol, da in hyp3_stacks.items():
    try:
        stack_lib.validate_stack(da, hyp3_years, label=f'HyP3 {pol}')
        print(f'HyP3  {pol} : OK  ({len(hyp3_years)} epoch(s))')
    except ValueError as e:
        print(f'HyP3  {pol} : FAIL — {e}')
        errors.append(str(e))

if s2_stack is not None:
    try:
        stack_lib.validate_stack(s2_stack, s2_years, label='S2 indices')
        print(f'S2 indices : OK  ({len(s2_years)} epoch(s))')
    except ValueError as e:
        print(f'S2 indices : FAIL — {e}')
        errors.append(str(e))

if errors:
    raise RuntimeError(f'{len(errors)} validation error(s):\n' + '\n'.join(errors))

print('\nAll stacks validated.')

## Save stacks as NetCDF

In [ ]:
# ── Cell 8: save all stacks to data/stacks/ ───────────────────────────────────────
STACKS_DIR.mkdir(parents=True, exist_ok=True)

saved = []

for pol, da in opera_stacks.items():
    path = STACKS_DIR / f'opera_{pol}.nc'
    stack_lib.save_stack(da, path)
    print(f'Saved: {path.name}')
    saved.append(path)

for pol, da in hyp3_stacks.items():
    path = STACKS_DIR / f'hyp3_{pol}.nc'
    stack_lib.save_stack(da, path)
    print(f'Saved: {path.name}')
    saved.append(path)

if s2_stack is not None:
    path = STACKS_DIR / 's2_indices.nc'
    stack_lib.save_stack(s2_stack, path)
    print(f'Saved: {path.name}')
    saved.append(path)

if s2_at_opera is not None:
    path = STACKS_DIR / 's2_at_opera.nc'
    stack_lib.save_stack(s2_at_opera, path)
    print(f'Saved: {path.name}')
    saved.append(path)

if s2_at_hyp3 is not None:
    path = STACKS_DIR / 's2_at_hyp3.nc'
    stack_lib.save_stack(s2_at_hyp3, path)
    print(f'Saved: {path.name}')
    saved.append(path)

print(f'\n{len(saved)} stack file(s) written to {STACKS_DIR}')

## Preview — baseline vs. latest epoch

Greyscale plots for the first SAR sensor with data. Confirms that the stack loaded
correctly and that spatial coverage is consistent across epochs.

In [ ]:
# ── Cell 9: preview baseline and latest epoch ─────────────────────────────────────
stacks   = opera_stacks if opera_stacks else hyp3_stacks
sensor_l = 'OPERA' if opera_stacks else 'HyP3'
years_l  = opera_years if opera_stacks else hyp3_years

if not stacks:
    print('No SAR stacks available — run cells above first.')
else:
    pol = pols[0]
    da  = stacks[pol]

    show_years = sorted({years_l[0], years_l[-1]})   # first and last
    n = len(show_years)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, year in zip(axes, show_years):
        sl   = da.sel(time=str(year), method='nearest')
        vals = sl.values
        vmin, vmax = np.nanpercentile(vals, [2, 98])
        ax.imshow(vals, cmap='gray', vmin=vmin, vmax=vmax)
        label = f'{year} (baseline)' if year == baseline else str(year)
        ax.set_title(f'{sensor_l} {pol} — {label}')
        ax.axis('off')

    plt.suptitle(SITE, fontsize=12)
    plt.tight_layout()
    OUT_FIGS.mkdir(parents=True, exist_ok=True)
    save_path = OUT_FIGS / f'04_{sensor_l.lower()}_{pol}_stack_preview.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')